<h3 style="color:#6FA8DC; font-weight:bold">04 — Outlier Detection using Percentile / Quantile Method</h3>

This notebook explains how percentiles can be used to identify extreme observations and how the same limits can be used for **capping**.

Topics:
- Percentile intuition
- Quantiles in pandas
- 1st/99th percentile
- 5th/95th percentile
- Detecting extreme observations
- Removing vs capping
- Choosing percentile limits
- Realistic examples
- Reusable functions
- Train/test leakage prevention

<h5 style="color:#78B89A; font-weight:bold;">What is a Percentile? → simple meaning</h5>

A percentile tells us the position of a value relative to the rest of the data.

For example:

```text
50th percentile → median
25th percentile → Q1
75th percentile → Q3
```

If a value is above the 99th percentile, it belongs to roughly the highest 1% of observations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

data = np.random.lognormal(mean=10.5, sigma=0.5, size=500)

df = pd.DataFrame({"income": data})

df.head()

<h5 style="color:#78B89A; font-weight:bold;">Calculate percentiles</h5>

In [ ]:
percentiles = df["income"].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])

percentiles

<h5 style="color:#78B89A; font-weight:bold;">1st and 99th percentile method</h5>

A common strategy is:

```text
Below 1st percentile → potential extreme low values
Above 99th percentile → potential extreme high values
```

This is especially useful when the business wants to control the most extreme tails.

In [ ]:
lower = df["income"].quantile(0.01)
upper = df["income"].quantile(0.99)

df["is_outlier"] = (
    (df["income"] < lower) |
    (df["income"] > upper)
)

print("1st percentile:", lower)
print("99th percentile:", upper)
print("Potential extreme observations:", df["is_outlier"].sum())

In [ ]:
outliers = df[df["is_outlier"]]
display(outliers.head(10))

<h5 style="color:#78B89A; font-weight:bold;">5th and 95th percentile</h5>

The threshold is a business/statistical choice.

For a more aggressive treatment:

```text
Below 5th percentile
Above 95th percentile
```

For a less aggressive treatment:

```text
Below 1st percentile
Above 99th percentile
```

In [ ]:
lower_5 = df["income"].quantile(0.05)
upper_95 = df["income"].quantile(0.95)

count_5_95 = (
    (df["income"] < lower_5) |
    (df["income"] > upper_95)
).sum()

print("5th percentile:", lower_5)
print("95th percentile:", upper_95)
print("Potential extreme observations:", count_5_95)

<h5 style="color:#78B89A; font-weight:bold;">Percentile Capping → often useful</h5>

Instead of deleting rows:

```text
value < lower limit → replace with lower limit
value > upper limit → replace with upper limit
```

This keeps the number of observations unchanged.

In [ ]:
df["income_capped"] = df["income"].clip(
    lower=lower,
    upper=upper
)

df[["income", "income_capped"]].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x=df["income"], ax=axes[0])
axes[0].set_title("Before Percentile Capping")

sns.boxplot(x=df["income_capped"], ax=axes[1])
axes[1].set_title("After Percentile Capping")

plt.tight_layout()
plt.show()

<h5 style="color:#78B89A; font-weight:bold;">Percentile limits as business rules</h5>

Suppose a company predicts delivery time.

If 99% of historical deliveries are below 10 days, values above the 99th percentile may need investigation.

But the correct threshold should depend on:
- domain knowledge
- cost of extreme values
- model objective
- amount of data
- whether extremes are genuine

<h5 style="color:#78B89A; font-weight:bold;">Reusable percentile capping function</h5>

In [ ]:
def percentile_cap(series, lower_percentile=0.01, upper_percentile=0.99):
    lower = series.quantile(lower_percentile)
    upper = series.quantile(upper_percentile)

    capped = series.clip(lower=lower, upper=upper)

    return capped, lower, upper

df["income_capped_1_99"], low, high = percentile_cap(
    df["income"],
    0.01,
    0.99
)

print("Lower limit:", low)
print("Upper limit:", high)

<h5 style="color:#78B89A; font-weight:bold;">Percentile vs IQR</h5>

| Percentile | IQR |
|---|---|
| User chooses percentile thresholds | Uses Q1/Q3 |
| Example: 1% and 99% | Example: 1.5 × IQR |
| Easy to control tail percentage | More rule-based |
| Useful for business-driven capping | Robust general-purpose method |

<h5 style="color:#78B89A; font-weight:bold;">Production / leakage rule ⭐</h5>

Percentile limits must be learned from the **training data**.

```text
X_train
   ↓
calculate 1st / 99th percentile
   ↓
save limits
   ↓
X_test / production data
   ↓
apply same limits
```

Do not calculate the limits from the complete dataset before splitting.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(
    df[["income"]],
    test_size=0.2,
    random_state=42
)

train_lower = X_train["income"].quantile(0.01)
train_upper = X_train["income"].quantile(0.99)

X_train["income_capped"] = X_train["income"].clip(
    train_lower, train_upper
)

X_test["income_capped"] = X_test["income"].clip(
    train_lower, train_upper
)

print("Learned training limits:")
print(train_lower, train_upper)

<h3 style="color:#6FA8DC; font-weight:bold">Percentile Revision</h3>

```text
Choose percentiles
      ↓
Calculate limits on training data
      ↓
Detect extreme values
      ↓
Investigate
   ↙        ↘
Remove     Cap / Keep
```

⭐ Percentile method is particularly useful when you explicitly want to control the **extreme tails** of a numerical feature.